# Feature Engineering & Anomaly Detection
## Bank Transactions — Fraud Detection Pipeline

---

### สิ่งที่เราพบจาก EDA (Recap)
| สัญญาณ | รายละเอียด |
|--------|------------|
| Amount Outliers | 4.75% ของธุรกรรมมียอดเกิน IQR upper bound |
| Device Sharing | 609 อุปกรณ์ถูกใช้กับหลายบัญชี (สูงสุด 9) |
| Login Attempts | 3.85% มีการ login 3-5 ครั้ง |
| Multi-Location | 266 บัญชีทำรายการใน 5+ เมือง |

### แผนการทำงานใน Notebook นี้
1. **Feature Engineering** — สร้างตัวแปรใหม่จาก insight ที่พบ
2. **Data Preprocessing** — Scale & Encode สำหรับ ML
3. **Anomaly Detection** — Isolation Forest, DBSCAN, Autoencoder
4. **Composite Risk Score** — รวมผลจากทุก Model
5. **Analysis & Interpretation** — วิเคราะห์ผลลัพธ์

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

df = pd.read_csv('bank_transactions.csv')
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], format='mixed')
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

---
## 2. Feature Engineering

สร้างตัวแปรใหม่ 4 กลุ่ม:
- **Account-Level Features** — พฤติกรรมระดับบัญชี
- **Transaction-Level Features** — ความผิดปกติของแต่ละธุรกรรม
- **Device & Location Features** — ความเสี่ยงด้านอุปกรณ์/สถานที่
- **Time-based Features** — รูปแบบเชิงเวลา

### 2.1 Account-Level Features
สร้างโปรไฟล์พฤติกรรม "ปกติ" ของแต่ละบัญชี

In [ ]:
# --- Account-Level Aggregation ---
account_stats = df.groupby('AccountID').agg(
    acct_txn_count=('TransactionID', 'count'),
    acct_mean_amount=('TransactionAmount', 'mean'),
    acct_std_amount=('TransactionAmount', 'std'),
    acct_max_amount=('TransactionAmount', 'max'),
    acct_median_amount=('TransactionAmount', 'median'),
    acct_mean_balance=('AccountBalance', 'mean'),
    acct_mean_duration=('TransactionDuration', 'mean'),
    acct_mean_login=('LoginAttempts', 'mean'),
    acct_unique_devices=('DeviceID', 'nunique'),
    acct_unique_locations=('Location', 'nunique'),
    acct_unique_merchants=('MerchantID', 'nunique'),
    acct_unique_channels=('Channel', 'nunique'),
    acct_unique_ips=('IP Address', 'nunique')
).reset_index()

# Fill std=NaN (accounts with 1 transaction) with 0
account_stats['acct_std_amount'] = account_stats['acct_std_amount'].fillna(0)

# Merge back to transaction-level
df = df.merge(account_stats, on='AccountID', how='left')

print(f'Account-level features added: {len(account_stats.columns)-1}')
account_stats.describe().round(2)

### 2.2 Transaction-Level Features
วัดว่าธุรกรรมนี้ "ผิดปกติ" แค่ไหนเมื่อเทียบกับพฤติกรรมปกติของบัญชี

In [ ]:
# --- Z-Score: ยอดเงินเทียบกับค่าเฉลี่ยของบัญชีนั้นๆ ---
df['amount_zscore'] = (df['TransactionAmount'] - df['acct_mean_amount']) / df['acct_std_amount'].replace(0, 1)

# --- Ratio: ยอดเงินเทียบกับยอดคงเหลือ ---
df['amount_to_balance_ratio'] = df['TransactionAmount'] / df['AccountBalance'].replace(0, 1)

# --- Ratio: ยอดเงินเทียบกับ Max ของบัญชี ---
df['amount_to_max_ratio'] = df['TransactionAmount'] / df['acct_max_amount'].replace(0, 1)

# --- Duration deviation: ระยะเวลาผิดปกติไหม ---
df['duration_deviation'] = abs(df['TransactionDuration'] - df['acct_mean_duration'])

# --- Login anomaly: login สูงกว่าค่าเฉลี่ยของบัญชีไหม ---
df['login_above_avg'] = (df['LoginAttempts'] > df['acct_mean_login']).astype(int)

# --- High login flag (3+) ---
df['high_login_flag'] = (df['LoginAttempts'] >= 3).astype(int)

# --- Amount outlier flag (IQR) ---
Q1 = df['TransactionAmount'].quantile(0.25)
Q3 = df['TransactionAmount'].quantile(0.75)
IQR = Q3 - Q1
df['amount_outlier_flag'] = (df['TransactionAmount'] > Q3 + 1.5 * IQR).astype(int)

print('Transaction-level features added: 7')
print(f"\nAmount Z-Score stats:")
print(df['amount_zscore'].describe().round(2))
print(f"\nAmount-to-Balance Ratio stats:")
print(df['amount_to_balance_ratio'].describe().round(2))

### 2.3 Device & Location Risk Features
วัดความเสี่ยงจากอุปกรณ์และสถานที่

In [ ]:
# --- Device shared count: อุปกรณ์นี้ถูกใช้กับกี่บัญชี ---
device_account_count = df.groupby('DeviceID')['AccountID'].nunique().reset_index()
device_account_count.columns = ['DeviceID', 'device_shared_accounts']
df = df.merge(device_account_count, on='DeviceID', how='left')

# --- Device is shared flag ---
df['device_shared_flag'] = (df['device_shared_accounts'] > 1).astype(int)

# --- IP shared count: IP นี้ถูกใช้กับกี่บัญชี ---
ip_account_count = df.groupby('IP Address')['AccountID'].nunique().reset_index()
ip_account_count.columns = ['IP Address', 'ip_shared_accounts']
df = df.merge(ip_account_count, on='IP Address', how='left')

# --- Merchant frequency per account: ไปร้านนี้บ่อยแค่ไหน ---
merchant_freq = df.groupby(['AccountID', 'MerchantID']).size().reset_index(name='merchant_visit_count')
df = df.merge(merchant_freq, on=['AccountID', 'MerchantID'], how='left')
df['is_new_merchant'] = (df['merchant_visit_count'] == 1).astype(int)

print('Device & Location features added: 5')
print(f"\nShared device stats:")
print(df['device_shared_accounts'].describe().round(2))
print(f"\nShared IP stats:")
print(df['ip_shared_accounts'].describe().round(2))

### 2.4 Time-based Features
รูปแบบเชิงเวลาที่อาจบ่งชี้ความผิดปกติ

In [ ]:
# --- Basic time features ---
df['hour'] = df['TransactionDate'].dt.hour
df['day_of_week'] = df['TransactionDate'].dt.dayofweek  # 0=Mon, 6=Sun
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['month'] = df['TransactionDate'].dt.month

# --- Transaction velocity: จำนวนธุรกรรมต่อวันของแต่ละบัญชี ---
df['txn_date'] = df['TransactionDate'].dt.date
daily_velocity = df.groupby(['AccountID', 'txn_date']).size().reset_index(name='daily_txn_count')
df = df.merge(daily_velocity, on=['AccountID', 'txn_date'], how='left')

# --- Time since last transaction per account ---
df = df.sort_values(['AccountID', 'TransactionDate'])
df['time_since_last_txn'] = df.groupby('AccountID')['TransactionDate'].diff().dt.total_seconds() / 3600  # hours
df['time_since_last_txn'] = df['time_since_last_txn'].fillna(-1)  # -1 = first transaction

# --- Rapid transaction flag: ธุรกรรมที่เกิดภายใน 1 ชั่วโมงจากครั้งก่อน ---
df['rapid_txn_flag'] = ((df['time_since_last_txn'] >= 0) & (df['time_since_last_txn'] < 1)).astype(int)

# Cleanup
df = df.drop(columns=['txn_date'])

print('Time-based features added: 7')
print(f"\nDaily transaction velocity:")
print(df['daily_txn_count'].describe().round(2))
print(f"\nTime since last txn (hours):")
print(df[df['time_since_last_txn'] >= 0]['time_since_last_txn'].describe().round(2))

### 2.5 Feature Summary

In [ ]:
# สรุป Features ที่สร้างทั้งหมด
original_cols = ['TransactionID', 'AccountID', 'TransactionAmount', 'TransactionDate',
                 'TransactionType', 'Location', 'DeviceID', 'IP Address', 'MerchantID',
                 'Channel', 'CustomerAge', 'CustomerOccupation', 'TransactionDuration',
                 'LoginAttempts', 'AccountBalance']
new_features = [c for c in df.columns if c not in original_cols]

print(f'Original columns: {len(original_cols)}')
print(f'New features created: {len(new_features)}')
print(f'Total columns: {len(df.columns)}')
print(f'\n--- New Features List ---')
for i, f in enumerate(new_features, 1):
    print(f'  {i:2d}. {f}')

print(f'\nDataset shape: {df.shape}')

---
## 3. Data Preprocessing สำหรับ Anomaly Detection

เลือก Features ที่เหมาะสมและ Scale ข้อมูล

In [ ]:
# เลือก Features สำหรับ Anomaly Detection
features_for_model = [
    # Transaction characteristics
    'TransactionAmount', 'TransactionDuration', 'LoginAttempts', 'AccountBalance',
    # Engineered: transaction-level
    'amount_zscore', 'amount_to_balance_ratio', 'amount_to_max_ratio',
    'duration_deviation', 'high_login_flag', 'amount_outlier_flag',
    # Engineered: device & location risk
    'device_shared_accounts', 'device_shared_flag', 'ip_shared_accounts', 'is_new_merchant',
    # Engineered: account profile
    'acct_txn_count', 'acct_std_amount', 'acct_unique_devices', 'acct_unique_locations',
    # Engineered: time-based
    'daily_txn_count', 'rapid_txn_flag', 'is_weekend',
    # Demographics
    'CustomerAge'
]

X = df[features_for_model].copy()

# ตรวจสอบ infinity / NaN
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

print(f'Features selected: {len(features_for_model)}')
print(f'Shape: {X.shape}')
X.describe().round(2)

In [ ]:
# Standard Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=features_for_model, index=X.index)

print('Scaling completed')
print(f'Mean (should be ~0): {X_scaled.mean().mean():.6f}')
print(f'Std (should be ~1): {X_scaled.std().mean():.6f}')

---
## 4. Anomaly Detection Models

### 4.1 Isolation Forest
หลักการ: แยก (isolate) ข้อมูลที่ผิดปกติออกจากกลุ่มได้เร็วกว่าข้อมูลปกติ

In [ ]:
# --- Isolation Forest ---
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,  # สมมติว่า ~5% เป็น anomaly
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

df['iso_forest_label'] = iso_forest.fit_predict(X_scaled)  # 1=normal, -1=anomaly
df['iso_forest_score'] = iso_forest.decision_function(X_scaled)  # lower = more anomalous

# Convert: -1 → 1 (anomaly), 1 → 0 (normal)
df['iso_forest_anomaly'] = (df['iso_forest_label'] == -1).astype(int)

n_anomaly = df['iso_forest_anomaly'].sum()
print(f'Isolation Forest Results:')
print(f'  Normal: {len(df) - n_anomaly:,} ({(len(df)-n_anomaly)/len(df)*100:.1f}%)')
print(f'  Anomaly: {n_anomaly:,} ({n_anomaly/len(df)*100:.1f}%)')
print(f'\nAnomaly Score Distribution:')
print(df['iso_forest_score'].describe().round(4))

In [ ]:
# Visualization: Isolation Forest Results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Score Distribution
axes[0].hist(df[df['iso_forest_anomaly']==0]['iso_forest_score'], bins=50, alpha=0.7, 
             color='steelblue', label='Normal', edgecolor='black')
axes[0].hist(df[df['iso_forest_anomaly']==1]['iso_forest_score'], bins=50, alpha=0.7, 
             color='red', label='Anomaly', edgecolor='black')
axes[0].set_title('Isolation Forest — Score Distribution')
axes[0].set_xlabel('Anomaly Score')
axes[0].legend()

# 2. Amount vs Balance (colored by anomaly)
normal = df[df['iso_forest_anomaly']==0]
anomaly = df[df['iso_forest_anomaly']==1]
axes[1].scatter(normal['TransactionAmount'], normal['AccountBalance'], 
                c='steelblue', alpha=0.1, s=5, label='Normal')
axes[1].scatter(anomaly['TransactionAmount'], anomaly['AccountBalance'], 
                c='red', alpha=0.5, s=15, label='Anomaly')
axes[1].set_title('Amount vs Balance — Anomalies Highlighted')
axes[1].set_xlabel('Transaction Amount')
axes[1].set_ylabel('Account Balance')
axes[1].legend()

# 3. Feature importance (mean anomaly score contribution)
anomaly_feature_means = df[df['iso_forest_anomaly']==1][features_for_model].mean()
normal_feature_means = df[df['iso_forest_anomaly']==0][features_for_model].mean()
diff = ((anomaly_feature_means - normal_feature_means) / normal_feature_means.replace(0,1) * 100).sort_values()
top_diff = pd.concat([diff.head(5), diff.tail(5)])
colors = ['red' if v > 0 else 'steelblue' for v in top_diff.values]
top_diff.plot(kind='barh', ax=axes[2], color=colors, edgecolor='black')
axes[2].set_title('Anomaly vs Normal — Feature Difference (%)')
axes[2].set_xlabel('% Difference')

plt.tight_layout()
plt.show()

### 4.2 DBSCAN
หลักการ: จัดกลุ่มข้อมูลที่อยู่ใกล้กัน ข้อมูลที่ไม่เข้ากลุ่มไหนเลย = Noise/Anomaly

In [ ]:
# ใช้ PCA ลดมิติก่อน เพื่อให้ DBSCAN ทำงานได้ดีขึ้น
pca = PCA(n_components=5, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'PCA: {X_scaled.shape[1]} features → {X_pca.shape[1]} components')
print(f'Explained variance ratio: {pca.explained_variance_ratio_.round(3)}')
print(f'Total variance explained: {pca.explained_variance_ratio_.sum():.1%}')

In [ ]:
# --- DBSCAN ---
dbscan = DBSCAN(
    eps=2.5,
    min_samples=10,
    n_jobs=-1
)

df['dbscan_label'] = dbscan.fit_predict(X_pca)
df['dbscan_anomaly'] = (df['dbscan_label'] == -1).astype(int)  # -1 = noise/anomaly

n_clusters = len(set(df['dbscan_label'])) - (1 if -1 in df['dbscan_label'].values else 0)
n_noise = df['dbscan_anomaly'].sum()

print(f'DBSCAN Results:')
print(f'  Clusters found: {n_clusters}')
print(f'  Normal (in clusters): {len(df) - n_noise:,} ({(len(df)-n_noise)/len(df)*100:.1f}%)')
print(f'  Noise/Anomaly: {n_noise:,} ({n_noise/len(df)*100:.1f}%)')
print(f'\nCluster distribution:')
print(df['dbscan_label'].value_counts().sort_index())

In [ ]:
# Visualization: DBSCAN — PCA 2D projection
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# PCA 2D view with clusters
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=df['dbscan_label'], 
                          cmap='tab10', alpha=0.3, s=5)
axes[0].set_title('DBSCAN Clusters (PCA 2D)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Normal vs Anomaly
normal_db = df[df['dbscan_anomaly']==0]
anomaly_db = df[df['dbscan_anomaly']==1]
axes[1].scatter(X_pca[normal_db.index, 0], X_pca[normal_db.index, 1], 
                c='steelblue', alpha=0.1, s=5, label='Normal')
axes[1].scatter(X_pca[anomaly_db.index, 0], X_pca[anomaly_db.index, 1], 
                c='red', alpha=0.5, s=15, label='Anomaly')
axes[1].set_title('DBSCAN — Anomalies Highlighted')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.3 Autoencoder (Neural Network)
หลักการ: เรียนรู้การบีบอัดข้อมูลปกติ → ถ้า Reconstruction Error สูง = ผิดปกติ

In [ ]:
try:
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.optimizers import Adam
    HAS_TF = True
    print('TensorFlow loaded successfully')
except ImportError:
    HAS_TF = False
    print('TensorFlow not available — will use sklearn alternative (LocalOutlierFactor)')
    from sklearn.neighbors import LocalOutlierFactor

In [ ]:
if HAS_TF:
    # --- Build Autoencoder ---
    input_dim = X_scaled.shape[1]
    encoding_dim = 8

    input_layer = Input(shape=(input_dim,))
    # Encoder
    encoded = Dense(64, activation='relu')(input_layer)
    encoded = BatchNormalization()(encoded)
    encoded = Dropout(0.2)(encoded)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = BatchNormalization()(encoded)
    encoded = Dense(encoding_dim, activation='relu')(encoded)
    # Decoder
    decoded = Dense(32, activation='relu')(encoded)
    decoded = BatchNormalization()(decoded)
    decoded = Dropout(0.2)(decoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

    print(f'Autoencoder: {input_dim} → 64 → 32 → {encoding_dim} → 32 → 64 → {input_dim}')
    autoencoder.summary()
else:
    print('Skipping Autoencoder — will use LOF instead')

In [ ]:
if HAS_TF:
    # --- Train Autoencoder ---
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = autoencoder.fit(
        X_scaled, X_scaled,
        epochs=50,
        batch_size=256,
        shuffle=True,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    # Plot training history
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history.history['loss'], label='Train Loss')
    ax.plot(history.history['val_loss'], label='Val Loss')
    ax.set_title('Autoencoder Training Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Skipped')

In [ ]:
if HAS_TF:
    # --- Calculate Reconstruction Error ---
    X_pred = autoencoder.predict(X_scaled, verbose=0)
    reconstruction_error = np.mean(np.square(X_scaled.values - X_pred), axis=1)
    df['ae_reconstruction_error'] = reconstruction_error

    # Threshold: 95th percentile
    threshold = np.percentile(reconstruction_error, 95)
    df['ae_anomaly'] = (reconstruction_error > threshold).astype(int)

    n_ae_anomaly = df['ae_anomaly'].sum()
    print(f'Autoencoder Results:')
    print(f'  Threshold (95th pctl): {threshold:.4f}')
    print(f'  Normal: {len(df)-n_ae_anomaly:,} ({(len(df)-n_ae_anomaly)/len(df)*100:.1f}%)')
    print(f'  Anomaly: {n_ae_anomaly:,} ({n_ae_anomaly/len(df)*100:.1f}%)')
else:
    # --- Fallback: Local Outlier Factor ---
    lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, n_jobs=-1)
    lof_labels = lof.fit_predict(X_scaled)
    df['ae_reconstruction_error'] = -lof.negative_outlier_factor_  # higher = more anomalous
    df['ae_anomaly'] = (lof_labels == -1).astype(int)

    n_ae_anomaly = df['ae_anomaly'].sum()
    print(f'Local Outlier Factor Results (fallback):')
    print(f'  Normal: {len(df)-n_ae_anomaly:,} ({(len(df)-n_ae_anomaly)/len(df)*100:.1f}%)')
    print(f'  Anomaly: {n_ae_anomaly:,} ({n_ae_anomaly/len(df)*100:.1f}%)')

In [ ]:
# Visualization: Reconstruction Error Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_name = 'Autoencoder' if HAS_TF else 'LOF'

axes[0].hist(df[df['ae_anomaly']==0]['ae_reconstruction_error'], bins=50, alpha=0.7,
             color='steelblue', label='Normal', edgecolor='black')
axes[0].hist(df[df['ae_anomaly']==1]['ae_reconstruction_error'], bins=50, alpha=0.7,
             color='red', label='Anomaly', edgecolor='black')
axes[0].set_title(f'{model_name} — Error Distribution')
axes[0].set_xlabel('Reconstruction Error' if HAS_TF else 'LOF Score')
axes[0].legend()

# Scatter: PCA view
normal_ae = df[df['ae_anomaly']==0]
anomaly_ae = df[df['ae_anomaly']==1]
axes[1].scatter(X_pca[normal_ae.index, 0], X_pca[normal_ae.index, 1],
                c='steelblue', alpha=0.1, s=5, label='Normal')
axes[1].scatter(X_pca[anomaly_ae.index, 0], X_pca[anomaly_ae.index, 1],
                c='red', alpha=0.5, s=15, label='Anomaly')
axes[1].set_title(f'{model_name} — Anomalies in PCA Space')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 5. Composite Risk Score

รวมผลจากทุก Model เข้าด้วยกันเป็น **Risk Score** เดียว

In [ ]:
# --- Normalize anomaly scores to 0-1 range ---
from sklearn.preprocessing import MinMaxScaler

# Isolation Forest: lower score = more anomalous → invert
df['iso_score_norm'] = MinMaxScaler().fit_transform(-df[['iso_forest_score']])

# Autoencoder/LOF: higher = more anomalous
df['ae_score_norm'] = MinMaxScaler().fit_transform(df[['ae_reconstruction_error']])

# DBSCAN: binary (0 or 1)
df['dbscan_score_norm'] = df['dbscan_anomaly'].astype(float)

# --- Composite Risk Score (weighted average) ---
w_iso = 0.4    # Isolation Forest (ดีกับ high-dimensional data)
w_ae = 0.35    # Autoencoder/LOF (จับ pattern ซับซ้อน)
w_db = 0.25    # DBSCAN (density-based)

df['risk_score'] = (w_iso * df['iso_score_norm'] + 
                    w_ae * df['ae_score_norm'] + 
                    w_db * df['dbscan_score_norm'])

# Risk Level Categories
df['risk_level'] = pd.cut(df['risk_score'], 
                          bins=[0, 0.3, 0.5, 0.7, 1.0],
                          labels=['Low', 'Medium', 'High', 'Critical'],
                          include_lowest=True)

print('Risk Score Distribution:')
print(df['risk_score'].describe().round(4))
print(f'\nRisk Level Breakdown:')
risk_counts = df['risk_level'].value_counts().sort_index()
for level, count in risk_counts.items():
    print(f'  {level:10s}: {count:>6,} ({count/len(df)*100:.1f}%)')

In [ ]:
# Visualization: Risk Score
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Risk Score Distribution
axes[0].hist(df['risk_score'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(0.3, color='green', linestyle='--', label='Low/Medium boundary')
axes[0].axvline(0.5, color='orange', linestyle='--', label='Medium/High boundary')
axes[0].axvline(0.7, color='red', linestyle='--', label='High/Critical boundary')
axes[0].set_title('Composite Risk Score Distribution')
axes[0].set_xlabel('Risk Score')
axes[0].legend(fontsize=9)

# 2. Risk Level Pie
colors_pie = ['#22c55e', '#f59e0b', '#ef4444', '#7f1d1d']
risk_counts.plot(kind='pie', ax=axes[1], colors=colors_pie, autopct='%1.1f%%',
                 startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Risk Level Distribution')
axes[1].set_ylabel('')

# 3. Model Agreement: Venn-like comparison
agreement = pd.DataFrame({
    'Isolation Forest': df['iso_forest_anomaly'],
    'DBSCAN': df['dbscan_anomaly'],
    'AE/LOF': df['ae_anomaly']
})
df['models_flagged'] = agreement.sum(axis=1)
flag_counts = df['models_flagged'].value_counts().sort_index()
flag_counts.plot(kind='bar', ax=axes[2], color=['#22c55e', '#f59e0b', '#ef4444', '#7f1d1d'],
                 edgecolor='black')
axes[2].set_title('Number of Models Flagging as Anomaly')
axes[2].set_xlabel('Models Agreed')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('\nModel Agreement:')
for n, count in flag_counts.items():
    print(f'  {int(n)} models flagged: {count:>6,} ({count/len(df)*100:.1f}%)')

---
## 6. Deep Dive — High Risk Transactions

In [ ]:
# ธุรกรรมที่ทุก Model เห็นตรงกันว่าผิดปกติ
consensus_anomalies = df[df['models_flagged'] >= 2].copy()
print(f'Transactions flagged by 2+ models: {len(consensus_anomalies):,}')
print(f'Transactions flagged by all 3 models: {(df["models_flagged"]==3).sum():,}')

# เปรียบเทียบ profile ของ anomaly vs normal
compare_cols = ['TransactionAmount', 'AccountBalance', 'TransactionDuration', 
                'LoginAttempts', 'device_shared_accounts', 'amount_zscore',
                'daily_txn_count', 'acct_unique_devices', 'acct_unique_locations']

comparison = pd.DataFrame({
    'Normal (mean)': df[df['models_flagged']==0][compare_cols].mean(),
    'Anomaly 2+ (mean)': consensus_anomalies[compare_cols].mean(),
    'Difference %': ((consensus_anomalies[compare_cols].mean() - df[df['models_flagged']==0][compare_cols].mean()) 
                     / df[df['models_flagged']==0][compare_cols].mean() * 100)
}).round(2)

print('\nAnomaly vs Normal Profile Comparison:')
comparison

In [ ]:
# Top 20 riskiest transactions
top_risk = df.nlargest(20, 'risk_score')[[
    'TransactionID', 'AccountID', 'TransactionAmount', 'TransactionType',
    'Channel', 'LoginAttempts', 'device_shared_accounts', 'amount_zscore',
    'risk_score', 'risk_level', 'models_flagged'
]].reset_index(drop=True)

print('Top 20 Riskiest Transactions:')
top_risk

In [ ]:
# Risk Level by various dimensions
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Risk by Channel
risk_channel = df.groupby('Channel')['risk_score'].mean().sort_values(ascending=False)
risk_channel.plot(kind='bar', ax=axes[0,0], color='steelblue', edgecolor='black')
axes[0,0].set_title('Average Risk Score by Channel')
axes[0,0].set_ylabel('Avg Risk Score')
axes[0,0].tick_params(axis='x', rotation=0)

# 2. Risk by Transaction Type
risk_type = df.groupby('TransactionType')['risk_score'].mean().sort_values(ascending=False)
risk_type.plot(kind='bar', ax=axes[0,1], color='coral', edgecolor='black')
axes[0,1].set_title('Average Risk Score by Transaction Type')
axes[0,1].set_ylabel('Avg Risk Score')
axes[0,1].tick_params(axis='x', rotation=0)

# 3. Risk by Occupation
risk_occ = df.groupby('CustomerOccupation')['risk_score'].mean().sort_values(ascending=False)
risk_occ.plot(kind='bar', ax=axes[1,0], color='mediumpurple', edgecolor='black')
axes[1,0].set_title('Average Risk Score by Occupation')
axes[1,0].set_ylabel('Avg Risk Score')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Risk Score vs Transaction Amount scatter
scatter = axes[1,1].scatter(df['TransactionAmount'], df['risk_score'], 
                            c=df['LoginAttempts'], cmap='YlOrRd', alpha=0.3, s=5)
axes[1,1].set_title('Risk Score vs Amount (color=LoginAttempts)')
axes[1,1].set_xlabel('Transaction Amount')
axes[1,1].set_ylabel('Risk Score')
plt.colorbar(scatter, ax=axes[1,1], label='Login Attempts')

plt.tight_layout()
plt.show()

---
## 7. Export Results

In [ ]:
# Export: เพิ่ม risk score กลับเข้าไปในข้อมูลต้นฉบับ
export_cols = [
    'TransactionID', 'AccountID', 'TransactionAmount', 'TransactionDate',
    'TransactionType', 'Location', 'DeviceID', 'Channel',
    'CustomerAge', 'CustomerOccupation', 'TransactionDuration', 'LoginAttempts', 'AccountBalance',
    # Key engineered features
    'amount_zscore', 'amount_to_balance_ratio', 'device_shared_accounts', 'daily_txn_count',
    'rapid_txn_flag', 'high_login_flag', 'amount_outlier_flag',
    # Model results
    'iso_forest_anomaly', 'dbscan_anomaly', 'ae_anomaly',
    'risk_score', 'risk_level', 'models_flagged'
]

df_export = df[export_cols].copy()
df_export.to_csv('bank_transactions_with_risk_scores.csv', index=False)

print(f'Exported: bank_transactions_with_risk_scores.csv')
print(f'Shape: {df_export.shape}')
print(f'\nRisk Level Summary:')
for level in ['Low', 'Medium', 'High', 'Critical']:
    count = (df_export['risk_level'] == level).sum()
    print(f'  {level:10s}: {count:>6,} ({count/len(df_export)*100:.1f}%)')

---
## 8. Summary & Conclusions

### Feature Engineering
สร้างตัวแปรใหม่ทั้งหมด ~30 ตัว แบ่งเป็น 4 กลุ่ม:
- **Account-Level** — โปรไฟล์พฤติกรรมของแต่ละบัญชี
- **Transaction-Level** — Z-score, Ratio, Flags ของแต่ละธุรกรรม
- **Device & Location** — ความเสี่ยงจากอุปกรณ์/สถานที่ที่แชร์กัน
- **Time-based** — Velocity, Rapid transaction, Weekend

### Anomaly Detection Models
| Model | Approach | จุดแข็ง |
|-------|----------|--------|
| Isolation Forest | Tree-based isolation | เร็ว, จัดการ high-dimensional ได้ดี |
| DBSCAN | Density-based clustering | จับกลุ่มที่ไม่เข้าพวก |
| Autoencoder/LOF | Reconstruction error | จับ pattern ที่ซับซ้อน |

### Composite Risk Score
- รวมผลจากทุก Model ด้วย Weighted Average
- แบ่งเป็น 4 ระดับ: Low / Medium / High / Critical
- ธุรกรรมที่ถูก flag โดย 2+ models มีความน่าเชื่อถือสูงที่สุด

### Next Steps
1. **Rule Engine** — สร้างกฎจาก pattern ที่พบเพื่อ flag แบบ real-time
2. **Dashboard** — Monitoring dashboard สำหรับ operation team
3. **Feedback Loop** — รับ feedback จาก analyst เพื่อปรับปรุง model
4. **A/B Testing** — เทียบ performance ของ model กับ rule-based เดิม